In [ ]:
# Import necessary libraries
import torch
import re
import os
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

# --- Ensure NLTK data is available (run once if needed) ---
def download_nltk_data():
    """Downloads necessary NLTK data packages."""
    packages = ['punkt', 'stopwords', 'wordnet', 'punkt_tab'] # Include 'punkt_tab'
    for package in packages:
        try:
            nltk.data.find(f'corpora/{package}' if package != 'punkt' else f'tokenizers/{package}')
        except LookupError:
            print(f"Downloading NLTK package: {package}...")
            nltk.download(package, quiet=True)
    print("NLTK data check complete.")

download_nltk_data()

# --- Configuration ---
# !! IMPORTANT !! Make sure this matches the directory where you saved
# the star-rating-trained model in the previous script.
try:
    # If running in Colab and saved to Drive
    from google.colab import drive
    # Ensure Drive is mounted if needed (might not be necessary if already mounted)
    try:
        drive.mount('/content/drive', force_remount=True)
    except: # Handle cases where drive might already be mounted or not in Colab
        pass
    model_save_directory = '/content/drive/My Drive/Colab Notebooks/SentimentModels_Stars'
    print("Assuming Colab environment for model path.")
except ModuleNotFoundError:
    # If running locally
    model_save_directory = './SentimentModels_Stars'
    print("Assuming local environment for model path.")

print(f"Attempting to load model from: {model_save_directory}")

# Check for GPU
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
print(f"Using device: {device}")

# Define the mapping from integer labels (used during training) to sentiment strings
# Must match the mapping used when training the star-based model:
# 0: Negative (1-2 Stars), 1: Neutral (3 Stars), 2: Positive (4-5 Stars)
label_map = {
    0: 'Negative',
    1: 'Neutral',
    2: 'Positive'
}

# --- Text Preprocessing Function (Same as before) ---
lemmatizer = WordNetLemmatizer()
stop_words_set = set(stopwords.words('english'))

def preprocess_text(text):
    """Cleans and preprocesses text data."""
    if not isinstance(text, str):
        return ""
    text = text.lower() # Lowercase
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE) # Remove URLs
    text = re.sub(r'\@\w+|\#', '', text) # Remove mentions and hashtags
    text = re.sub(r'[^\w\s]', '', text) # Remove punctuation
    text = re.sub(r'\d+', '', text) # Remove numbers
    tokens = word_tokenize(text) # Tokenize
    processed_tokens = [
        lemmatizer.lemmatize(word) for word in tokens
        if word not in stop_words_set and word.isalpha() and len(word) > 1
    ]
    return ' '.join(processed_tokens)

# --- Load Model and Tokenizer ---
try:
    print("Loading fine-tuned tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(model_save_directory)
    print("Loading fine-tuned model...")
    model = AutoModelForSequenceClassification.from_pretrained(model_save_directory)
    model.to(device) # Move model to the appropriate device
    model.eval() # Set model to evaluation mode
    print("Model and tokenizer loaded successfully.")
    model_loaded = True
except OSError as e:
    print(f"Error loading model/tokenizer from {model_save_directory}: {e}")
    print("Please ensure the directory exists and contains the saved model files ('pytorch_model.bin', 'config.json', 'tokenizer_config.json', etc.).")
    print("Cannot proceed with predictions.")
    model_loaded = False
except Exception as e:
    print(f"An unexpected error occurred during loading: {e}")
    model_loaded = False

# --- Prediction Function ---
def predict_overall_sentiment(text, loaded_model, loaded_tokenizer):
    """Predicts overall sentiment for a given text using the star-rating model."""
    if not model_loaded:
         return "Error: Model not loaded."

    # 1. Preprocess the input text
    processed_text = preprocess_text(text)
    if not processed_text:
        return "Input text is empty after preprocessing."

    # 2. Tokenize
    # Adjust max_length if needed, should match training if possible
    inputs = loaded_tokenizer(
        processed_text,
        return_tensors='pt',
        truncation=True,
        padding=True,
        max_length=256 # Use the same MAX_LEN as during training if possible
    )

    # 3. Move inputs to the same device as the model
    inputs = {k: v.to(device) for k, v in inputs.items()}

    # 4. Predict
    with torch.no_grad(): # Disable gradient calculations for inference
        outputs = loaded_model(**inputs)
        logits = outputs.logits

    # 5. Get prediction index
    prediction_index = torch.argmax(logits, dim=-1).squeeze().item()

    # 6. Map index to sentiment label
    sentiment = label_map.get(prediction_index, "Unknown Label")

    return sentiment

# --- User Input Loop ---
if model_loaded:
    print("\n--- Overall Sentiment Predictor ---")
    print("Enter your review text below. Type 'quit' to exit.")

    while True:
        user_input = input("\nEnter review text: ")
        if user_input.lower() == 'quit':
            break
        if not user_input.strip():
            print("Please enter some text.")
            continue

        # Predict sentiment
        predicted_sentiment = predict_overall_sentiment(user_input, model, tokenizer)

        print(f"Predicted Sentiment: {predicted_sentiment}")

else:
    print("\nSkipping prediction loop as the model could not be loaded.")

print("\n--- Exiting Sentiment Predictor ---")

NLTK data check complete.
Mounted at /content/drive
Assuming Colab environment for model path.
Attempting to load model from: /content/drive/My Drive/Colab Notebooks/SentimentModels_Stars
Using device: cuda
Loading fine-tuned tokenizer...
Loading fine-tuned model...
Model and tokenizer loaded successfully.

--- Overall Sentiment Predictor ---
Enter your review text below. Type 'quit' to exit.
Predicted Sentiment: Positive
Predicted Sentiment: Negative
Predicted Sentiment: Positive
Predicted Sentiment: Positive
Predicted Sentiment: Negative
Predicted Sentiment: Positive
Predicted Sentiment: Negative
Predicted Sentiment: Positive
Predicted Sentiment: Positive
Predicted Sentiment: Negative
Predicted Sentiment: Positive
